# Student Mistake Tagging System

This notebook creates a flow to:
1. Read student answers from a Google Sheet
2. Use an LLM API to analyze mistakes for answers with marks < max marks
3. Generate precise mistake tags for each student error
4. Export results to a CSV file with mistake categories

## Setup and Installation
First, install required packages (run once):
```bash
pip install gspread oauth2client pandas openai anthropic
```

In [ ]:
# Import required libraries
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd
import json
from datetime import datetime
from typing import Dict, List, Optional
import time

## Configuration
Set up your API keys and Google Sheets credentials here.

In [ ]:
# Configuration
CONFIG = {
    # Google Sheets Configuration
    'GOOGLE_SHEET_NAME': 'Student Answers Sheet',  # Replace with your sheet name
    'CREDENTIALS_FILE': 'credentials.json',  # Path to your Google service account JSON
    
    # LLM API Configuration
    'LLM_PROVIDER': 'openai',  # Options: 'openai', 'anthropic', 'custom'
    'API_KEY': 'your-api-key-here',  # Replace with your actual API key
    'MODEL': 'gpt-4',  # Model to use (e.g., 'gpt-4', 'claude-3-opus-20240229')
    
    # Output Configuration
    'OUTPUT_CSV': f'student_mistakes_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
}

## Google Sheets Integration

### Setup Instructions:
1. Go to Google Cloud Console
2. Create a new project or select existing one
3. Enable Google Sheets API
4. Create Service Account credentials
5. Download the JSON key file and save as 'credentials.json'
6. Share your Google Sheet with the service account email

In [ ]:
def authenticate_google_sheets():
    """
    Authenticate with Google Sheets API using service account credentials.
    
    Returns:
        gspread.Client: Authenticated client object
    """
    scope = [
        'https://spreadsheets.google.com/feeds',
        'https://www.googleapis.com/auth/drive'
    ]
    
    try:
        credentials = ServiceAccountCredentials.from_json_keyfile_name(
            CONFIG['CREDENTIALS_FILE'], 
            scope
        )
        client = gspread.authorize(credentials)
        print("✓ Successfully authenticated with Google Sheets")
        return client
    except Exception as e:
        print(f"✗ Authentication failed: {str(e)}")
        raise

def read_student_data(client, sheet_name: str) -> pd.DataFrame:
    """
    Read student answer data from Google Sheets.
    
    Expected columns:
    - student_id: Unique identifier for student
    - student_name: Student's name
    - question_id: Question identifier
    - question_text: The actual question
    - student_answer: Student's submitted answer
    - model_solution: Correct answer/solution
    - explanation: Explanation of the solution
    - marks_awarded: Points received
    - max_marks: Maximum possible points
    
    Args:
        client: Authenticated Google Sheets client
        sheet_name: Name of the Google Sheet
        
    Returns:
        pd.DataFrame: Student answer data
    """
    try:
        sheet = client.open(sheet_name).sheet1
        data = sheet.get_all_records()
        df = pd.DataFrame(data)
        print(f"✓ Successfully loaded {len(df)} rows from Google Sheets")
        print(f"  Columns: {list(df.columns)}")
        return df
    except Exception as e:
        print(f"✗ Failed to read Google Sheets: {str(e)}")
        raise

## LLM API Integration

This section handles the LLM API calls for mistake analysis.

In [ ]:
# LLM Prompt Template
MISTAKE_ANALYSIS_PROMPT = """YOUR ROLE AND MISSION
You are an expert science teacher aligned with the Singapore Ministry of Education curriculum. Your mission is CRITICAL: to meticulously analyze EVERY SINGLE student answer that received less than full marks, identify the specific reasoning errors, and create precise, actionable mistake tags.

CRITICAL REQUIREMENT: You must examine each student answer individually and thoughtfully. Generic or rushed tagging will fail to help students improve. Each mistake deserves careful analysis.

TASK BREAKDOWN

STEP 1: Individual Answer Analysis (MANDATORY FOR EACH ROW)
For every single student answer where marks awarded < max marks:
1. READ the student's answer completely - don't skim
2. READ the model solution and explanation completely
3. IDENTIFY the specific point of divergence - where exactly did the student go wrong?
4. DIAGNOSE the root cause - was it:
   - A conceptual misunderstanding? (Which concept specifically?)
   - A procedural error? (Which step?)
   - A calculation mistake? (What type?)
   - A unit/notation error?
   - Incomplete reasoning?
   - Misreading the question?
   - Something else?
5. DOCUMENT your thinking: Before assigning a tag, write out (in your analysis): "Student did X, but should have done Y because Z"

STEP 2: Create Precise Mistake Tags

RULES FOR TAG CREATION:
✅ DO: Create tags that pinpoint the SPECIFIC cognitive error (e.g., "Confused mass-weight" not "wrong concept")
✅ DO: Analyze WHY the student made this mistake
✅ DO: Check if similar mistakes appear across multiple questions by this student
✅ DO: Reuse existing tags ONLY if the mistake truly matches the established definition
✅ DO: Keep tags concise (2-3 words) but meaningful

❌ NEVER: Use vague tags like "wrong answer", "incorrect", "wrong option"
❌ NEVER: Skip the analysis and jump to tagging
❌ NEVER: Assume the mistake without checking the student's actual work
❌ NEVER: Use the same tag repeatedly without verifying each instance
❌ NEVER: Create tags based on the question type rather than the student's error

TAG CATEGORIES FRAMEWORK
Use this framework to guide your analysis (create specific tags within these categories):

A. Conceptual Errors
   - Fundamental misunderstanding of a principle
   - Confusion between related concepts
   - Missing prerequisite knowledge
   Examples: "Force-energy confusion", "Confuses series-parallel", "Density-mass mix-up"

B. Procedural Errors
   - Incorrect operation/formula applied
   - Steps performed in wrong order
   - Missing critical step
   Examples: "Division instead of multiplication", "Skipped normalization step", "Forgot to convert units"

C. Reasoning Errors
   - Logical fallacy in explanation
   - Incomplete justification
   - Correct process but wrong conclusion
   Examples: "Incomplete causal chain", "Reverse causation", "Correlation-causation error"

D. Execution Errors
   - Calculation mistakes (arithmetic)
   - Unit conversion errors
   - Notation errors
   Examples: "Decimal place error", "Unit not converted", "Sign error"

E. Question Interpretation Errors
   - Misread question requirement
   - Answered different question
   - Missed key constraint
   Examples: "Missed 'not' in question", "Calculated X not Y", "Ignored given condition"

F. Pattern-Based Classification
   - Misconception: Same conceptual error across ≥3 similar questions
   - Careless pattern: Correct approach but execution errors across ≥2 attempts
   - Complexity struggle: Errors primarily in multi-step or cross-concept questions

---

STUDENT ANSWER TO ANALYZE:

Question ID: {question_id}
Question: {question_text}

Student Answer: {student_answer}
Model Solution: {model_solution}
Explanation: {explanation}

Marks Awarded: {marks_awarded}/{max_marks}

---

Please provide your analysis in the following JSON format:
{{
    "analysis": "Your detailed analysis following the 'Student did X, but should have done Y because Z' format",
    "mistake_tag": "Concise, specific mistake tag (2-3 words)",
    "mistake_category": "One of: Conceptual Error, Procedural Error, Reasoning Error, Execution Error, Question Interpretation Error, Pattern-Based",
    "severity": "One of: Critical, Moderate, Minor",
    "remediation_suggestion": "Brief suggestion for how to address this mistake"
}}
"""

In [ ]:
def call_llm_api(prompt: str, provider: str = 'openai', api_key: str = None, model: str = 'gpt-4') -> str:
    """
    Call LLM API to analyze student mistakes.
    
    Args:
        prompt: The formatted prompt with student data
        provider: LLM provider ('openai', 'anthropic', or 'custom')
        api_key: API key for authentication
        model: Model identifier
        
    Returns:
        str: LLM response containing mistake analysis
    """
    # PLACEHOLDER: Replace with actual API implementation
    # This is where you'll integrate your chosen LLM API
    
    if provider == 'openai':
        # Example OpenAI implementation (uncomment and configure when ready)
        # import openai
        # openai.api_key = api_key
        # response = openai.ChatCompletion.create(
        #     model=model,
        #     messages=[
        #         {"role": "system", "content": "You are an expert science teacher."},
        #         {"role": "user", "content": prompt}
        #     ],
        #     temperature=0.3,
        #     response_format={"type": "json_object"}
        # )
        # return response.choices[0].message.content
        pass
    
    elif provider == 'anthropic':
        # Example Anthropic implementation (uncomment and configure when ready)
        # import anthropic
        # client = anthropic.Anthropic(api_key=api_key)
        # message = client.messages.create(
        #     model=model,
        #     max_tokens=1024,
        #     messages=[
        #         {"role": "user", "content": prompt}
        #     ]
        # )
        # return message.content[0].text
        pass
    
    elif provider == 'custom':
        # Add your custom API implementation here
        pass
    
    # For testing: Return a mock response
    print("⚠ Using placeholder LLM response (configure API for production use)")
    return json.dumps({
        "analysis": "Student did not convert units from cm to m, but should have converted because the formula requires SI units",
        "mistake_tag": "Forgot unit conversion",
        "mistake_category": "Procedural Error",
        "severity": "Moderate",
        "remediation_suggestion": "Review unit conversion protocols and SI unit requirements"
    })

def analyze_student_mistake(row: Dict) -> Dict:
    """
    Analyze a single student answer and generate mistake tags.
    
    Args:
        row: Dictionary containing student answer data
        
    Returns:
        Dict: Analysis results including mistake tag and category
    """
    # Format the prompt with student data
    formatted_prompt = MISTAKE_ANALYSIS_PROMPT.format(
        question_id=row.get('question_id', 'N/A'),
        question_text=row.get('question_text', ''),
        student_answer=row.get('student_answer', ''),
        model_solution=row.get('model_solution', ''),
        explanation=row.get('explanation', ''),
        marks_awarded=row.get('marks_awarded', 0),
        max_marks=row.get('max_marks', 0)
    )
    
    # Call LLM API
    try:
        response = call_llm_api(
            formatted_prompt,
            provider=CONFIG['LLM_PROVIDER'],
            api_key=CONFIG['API_KEY'],
            model=CONFIG['MODEL']
        )
        
        # Parse JSON response
        result = json.loads(response)
        return result
        
    except json.JSONDecodeError as e:
        print(f"✗ Failed to parse LLM response: {str(e)}")
        return {
            "analysis": "Error parsing response",
            "mistake_tag": "Analysis Error",
            "mistake_category": "Unknown",
            "severity": "Unknown",
            "remediation_suggestion": "Manual review required"
        }
    except Exception as e:
        print(f"✗ Error during analysis: {str(e)}")
        return {
            "analysis": f"Error: {str(e)}",
            "mistake_tag": "API Error",
            "mistake_category": "Unknown",
            "severity": "Unknown",
            "remediation_suggestion": "Check API configuration"
        }

## Main Processing Pipeline

Process all student answers and generate mistake tags.

In [ ]:
def process_student_answers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Process all student answers and generate mistake tags.
    Only analyzes answers where marks_awarded < max_marks.
    
    Args:
        df: DataFrame with student answer data
        
    Returns:
        pd.DataFrame: Original data with added mistake analysis columns
    """
    print("\n" + "="*60)
    print("STARTING MISTAKE ANALYSIS")
    print("="*60)
    
    # Initialize new columns
    df['mistake_analysis'] = ''
    df['mistake_tag'] = ''
    df['mistake_category'] = ''
    df['severity'] = ''
    df['remediation_suggestion'] = ''
    
    # Filter rows that need analysis (marks < max marks)
    needs_analysis = df['marks_awarded'] < df['max_marks']
    rows_to_analyze = df[needs_analysis].index
    
    print(f"\nTotal rows: {len(df)}")
    print(f"Rows needing analysis (marks < max): {len(rows_to_analyze)}")
    print(f"Rows with full marks: {len(df) - len(rows_to_analyze)}")
    print("\n" + "-"*60)
    
    # Process each row that needs analysis
    for idx, row_idx in enumerate(rows_to_analyze, 1):
        row = df.loc[row_idx]
        
        print(f"\nAnalyzing [{idx}/{len(rows_to_analyze)}]: Student {row.get('student_id', 'N/A')} - Question {row.get('question_id', 'N/A')}")
        print(f"  Marks: {row.get('marks_awarded', 0)}/{row.get('max_marks', 0)}")
        
        # Analyze the mistake
        analysis_result = analyze_student_mistake(row.to_dict())
        
        # Update DataFrame with results
        df.at[row_idx, 'mistake_analysis'] = analysis_result.get('analysis', '')
        df.at[row_idx, 'mistake_tag'] = analysis_result.get('mistake_tag', '')
        df.at[row_idx, 'mistake_category'] = analysis_result.get('mistake_category', '')
        df.at[row_idx, 'severity'] = analysis_result.get('severity', '')
        df.at[row_idx, 'remediation_suggestion'] = analysis_result.get('remediation_suggestion', '')
        
        print(f"  ✓ Tag: {analysis_result.get('mistake_tag', 'N/A')}")
        print(f"  ✓ Category: {analysis_result.get('mistake_category', 'N/A')}")
        
        # Rate limiting (adjust as needed for your API)
        if idx < len(rows_to_analyze):  # Don't sleep after last item
            time.sleep(0.5)  # Small delay to avoid rate limits
    
    print("\n" + "="*60)
    print("ANALYSIS COMPLETE")
    print("="*60)
    
    return df

## Export Results

Export the analyzed data to CSV.

In [ ]:
def export_to_csv(df: pd.DataFrame, output_file: str) -> None:
    """
    Export analyzed data to CSV file.
    
    Args:
        df: DataFrame with analysis results
        output_file: Path to output CSV file
    """
    try:
        df.to_csv(output_file, index=False, encoding='utf-8')
        print(f"\n✓ Results exported to: {output_file}")
        print(f"  Total rows: {len(df)}")
        print(f"  Columns: {len(df.columns)}")
        
        # Print summary statistics
        print("\n" + "="*60)
        print("SUMMARY STATISTICS")
        print("="*60)
        
        mistakes_analyzed = df['mistake_tag'].notna() & (df['mistake_tag'] != '')
        print(f"\nTotal mistakes analyzed: {mistakes_analyzed.sum()}")
        
        if mistakes_analyzed.sum() > 0:
            print("\nTop 10 Mistake Tags:")
            tag_counts = df[mistakes_analyzed]['mistake_tag'].value_counts().head(10)
            for tag, count in tag_counts.items():
                print(f"  - {tag}: {count}")
            
            print("\nMistake Categories:")
            category_counts = df[mistakes_analyzed]['mistake_category'].value_counts()
            for category, count in category_counts.items():
                print(f"  - {category}: {count}")
            
            print("\nSeverity Distribution:")
            severity_counts = df[mistakes_analyzed]['severity'].value_counts()
            for severity, count in severity_counts.items():
                print(f"  - {severity}: {count}")
        
    except Exception as e:
        print(f"✗ Failed to export CSV: {str(e)}")
        raise

## Main Execution

Run the complete pipeline.

In [ ]:
def main():
    """
    Main execution function - runs the complete pipeline.
    """
    print("\n" + "#"*60)
    print("# STUDENT MISTAKE TAGGING SYSTEM")
    print("#"*60)
    
    try:
        # Step 1: Authenticate with Google Sheets
        print("\n[1/4] Authenticating with Google Sheets...")
        client = authenticate_google_sheets()
        
        # Step 2: Read student data
        print("\n[2/4] Reading student data...")
        df = read_student_data(client, CONFIG['GOOGLE_SHEET_NAME'])
        
        # Step 3: Process and analyze mistakes
        print("\n[3/4] Processing student answers...")
        df_analyzed = process_student_answers(df)
        
        # Step 4: Export results
        print("\n[4/4] Exporting results...")
        export_to_csv(df_analyzed, CONFIG['OUTPUT_CSV'])
        
        print("\n" + "#"*60)
        print("# PIPELINE COMPLETED SUCCESSFULLY")
        print("#"*60)
        
        return df_analyzed
        
    except Exception as e:
        print(f"\n✗ Pipeline failed: {str(e)}")
        raise

# Execute the pipeline
if __name__ == "__main__":
    results_df = main()

## Optional: Test with Sample Data

Use this cell to test the pipeline with sample data before connecting to Google Sheets.

In [ ]:
# Sample data for testing (without Google Sheets)
def test_with_sample_data():
    """
    Test the pipeline with sample data.
    """
    sample_data = {
        'student_id': ['S001', 'S002', 'S003', 'S001'],
        'student_name': ['Alice Wong', 'Bob Tan', 'Charlie Lee', 'Alice Wong'],
        'question_id': ['Q1', 'Q1', 'Q2', 'Q3'],
        'question_text': [
            'Calculate the density of an object with mass 500g and volume 200cm³',
            'Calculate the density of an object with mass 500g and volume 200cm³',
            'Explain why objects sink or float in water',
            'Convert 50°C to Kelvin'
        ],
        'student_answer': [
            'Density = 500/200 = 2.5',
            'Density = mass/volume = 500g/200cm³ = 2.5 g/cm³',
            'Objects float because they are light',
            '50 + 273 = 323K'
        ],
        'model_solution': [
            'Density = mass/volume = 500g/200cm³ = 2.5 g/cm³',
            'Density = mass/volume = 500g/200cm³ = 2.5 g/cm³',
            'Objects float when their density is less than water (1 g/cm³). The buoyant force equals the weight of displaced water.',
            'K = °C + 273.15 = 50 + 273.15 = 323.15K'
        ],
        'explanation': [
            'Must include units in the answer',
            'Complete answer with proper units',
            'Must explain the density relationship, not just mass',
            'Kelvin conversion requires adding 273.15, not 273'
        ],
        'marks_awarded': [1, 2, 1, 1],
        'max_marks': [2, 2, 3, 2]
    }
    
    df = pd.DataFrame(sample_data)
    print("Sample data created:")
    print(df)
    
    # Process the sample data
    df_analyzed = process_student_answers(df)
    
    # Export results
    test_output = 'test_' + CONFIG['OUTPUT_CSV']
    export_to_csv(df_analyzed, test_output)
    
    return df_analyzed

# Uncomment to run test
# test_results = test_with_sample_data()

## Usage Instructions

### Setup Steps:

1. **Install Dependencies:**
   ```bash
   pip install gspread oauth2client pandas openai anthropic
   ```

2. **Configure Google Sheets:**
   - Create a service account in Google Cloud Console
   - Download credentials JSON file
   - Share your Google Sheet with the service account email
   - Update `CREDENTIALS_FILE` in the CONFIG cell

3. **Configure LLM API:**
   - Choose your LLM provider (OpenAI, Anthropic, or custom)
   - Add your API key to CONFIG
   - Uncomment the appropriate API implementation in `call_llm_api()`

4. **Prepare Your Google Sheet:**
   - Ensure it has these columns:
     - student_id
     - student_name
     - question_id
     - question_text
     - student_answer
     - model_solution
     - explanation
     - marks_awarded
     - max_marks

5. **Run the Pipeline:**
   - Execute all cells in order
   - Or run the `main()` function in the execution cell

### Testing:
- Use the sample data test function to verify everything works before processing real data
- Start with a small subset of your Google Sheet to test the integration

### Output:
- CSV file will be created with timestamp
- Contains all original columns plus:
  - mistake_analysis
  - mistake_tag
  - mistake_category
  - severity
  - remediation_suggestion